# OncoVision: train the true 3D brain tumor model (3D U-Net on BraTS 2023)

**How to run (important, so nothing gets lost):**
1. Right panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
2. Top right: **Save Version → Save & Run All (Commit) → Save**. Do *not* use the normal Run All.
3. Close the tab if you like. It runs in the background for about 4 hours.
4. Later open the notebook page → **Output** tab → download `oncovision_3d.zip` and send the printed scores to Claude.

Data: BraTS 2023 glioma training set (1,251 cases, CC-BY-4.0, public Hugging Face mirror, no login needed).

In [ ]:
!git clone -q -b vaishnavi/optimistic-noether-034odi https://github.com/Vaishnavi220506/Medical.git /kaggle/working/oncovision
%cd /kaggle/working/oncovision
!pip install -q -r requirements.txt huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings!")

## 1. Download BraTS 2023 (about 12 GB, kept in /tmp so it doesn't fill the output)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("MedOtter/brats2023-gli-dataset", repo_type="dataset", local_dir="/tmp/brats", max_workers=16,
                  allow_patterns=["ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/*"])
!ls /tmp/brats/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData | wc -l

## 2. Train (about 3-4 hours on a T4). 60 patients are held out: 40 for validation, 20 for a final test score.

In [ ]:
!python -m oncovision.training.train_segmenter3d --data /tmp/brats/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData \
    --prepared-dir /tmp/brats3d --epochs 60 --samples-per-epoch 400 --patch 128 --batch-size 2 \
    --val-cases 40 --test-cases 20 --val-every 5 --workers 4 --out weights/brain_mri/segmenter3d.pt

## 3. Scores and package

In [ ]:
import torch, json
meta = torch.load("weights/brain_mri/segmenter3d.pt", map_location="cpu", weights_only=True)["meta"]
print("SCORES (send these to Claude):", json.dumps(meta["metrics"]), "best epoch", meta["epoch"])
!python scripts/export_weights.py --src weights --dst export_weights
!cd /kaggle/working && zip -qj oncovision_3d.zip oncovision/export_weights/brain_mri/segmenter3d.pt && ls -la oncovision_3d.zip